In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!nvidia-smi

Wed Nov 18 16:50:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
import transformers
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

PyTorch: 2.11.0+cu130
Transformers: 5.20.2
Accelerate: 1.15.0
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [4]:
!pip install -q bitsandbytes

In [5]:
import bitsandbytes as bnb
from transformers import BitsAndBytesConfig

print("BitsAndBytes:", bnb.__version__)
print("4-bit quantization support: Available")

BitsAndBytes: 0.50.2
4-bit quantization support: Available


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Clear GPU memory
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading FP16 model...")

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cuda:0"
)

model_fp16.eval()

# Measure memory
fp16_memory = torch.cuda.memory_allocated(0) / (1024 ** 3)
fp16_reserved = torch.cuda.memory_reserved(0) / (1024 ** 3)

print("\nFP16 MODEL LOADED SUCCESSFULLY")
print("Model:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))
print(f"Allocated GPU memory: {fp16_memory:.3f} GB")
print(f"Reserved GPU memory: {fp16_reserved:.3f} GB")

Loading tokenizer...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


FP16 MODEL LOADED SUCCESSFULLY
Model: Qwen/Qwen2.5-0.5B-Instruct
GPU: Tesla T4
Allocated GPU memory: 0.920 GB
Reserved GPU memory: 0.934 GB


In [7]:
import time
import torch

prompt = "Explain artificial intelligence in simple words."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to("cuda:0")

# Warm-up
with torch.inference_mode():
    _ = model_fp16.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False
    )

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

# Actual benchmark
start_time = time.time()

with torch.inference_mode():
    outputs = model_fp16.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

torch.cuda.synchronize()
end_time = time.time()

generated_tokens = outputs.shape[1] - inputs["input_ids"].shape[1]
elapsed_time = end_time - start_time

print("FP16 BENCHMARK")
print("Generated tokens:", generated_tokens)
print("Time taken:", round(elapsed_time, 3), "seconds")
print("Tokens per second:", round(generated_tokens / elapsed_time, 2))
print("Peak GPU memory:", round(torch.cuda.max_memory_allocated(0) / 1024**3, 3), "GB")
print("\nGenerated response:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

FP16 BENCHMARK
Generated tokens: 100
Time taken: 2.81 seconds
Tokens per second: 35.59
Peak GPU memory: 0.933 GB

Generated response:
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain artificial intelligence in simple words.
assistant
Artificial Intelligence (AI) is a field of computer science that focuses on creating intelligent machines that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and understanding language. AI systems use algorithms and statistical models to analyze data and make predictions or decisions based on patterns and relationships within that data.

In simpler terms, AI involves the development of software programs that can learn from experience, recognize patterns, and respond to inputs in ways that mimic human behavior. This technology has


In [8]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Clear unused GPU memory
del model_fp16
torch.cuda.empty_cache()

# Configure 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

torch.cuda.reset_peak_memory_stats()

print("Loading 4-bit quantized model...")

model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="cuda:0"
)

model_4bit.eval()

quant_memory = torch.cuda.memory_allocated(0) / (1024 ** 3)
quant_reserved = torch.cuda.memory_reserved(0) / (1024 ** 3)

print("\n4-BIT MODEL LOADED SUCCESSFULLY")
print("Quantization: NF4")
print(f"Allocated GPU memory: {quant_memory:.3f} GB")
print(f"Reserved GPU memory: {quant_reserved:.3f} GB")

Loading 4-bit quantized model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


4-BIT MODEL LOADED SUCCESSFULLY
Quantization: NF4
Allocated GPU memory: 0.435 GB
Reserved GPU memory: 0.951 GB


In [9]:
import time
import torch

prompt = "Explain artificial intelligence in simple words."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to("cuda:0")

# Warm-up
with torch.inference_mode():
    _ = model_4bit.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False
    )

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

# Actual benchmark
start_time = time.time()

with torch.inference_mode():
    outputs_4bit = model_4bit.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

torch.cuda.synchronize()
end_time = time.time()

generated_tokens_4bit = (
    outputs_4bit.shape[1] - inputs["input_ids"].shape[1]
)

elapsed_time_4bit = end_time - start_time

print("4-BIT BENCHMARK")
print("Generated tokens:", generated_tokens_4bit)
print("Time taken:", round(elapsed_time_4bit, 3), "seconds")
print(
    "Tokens per second:",
    round(generated_tokens_4bit / elapsed_time_4bit, 2)
)
print(
    "Peak GPU memory:",
    round(torch.cuda.max_memory_allocated(0) / 1024**3, 3),
    "GB"
)

print("\nGenerated response:")
print(tokenizer.decode(outputs_4bit[0], skip_special_tokens=True))

4-BIT BENCHMARK
Generated tokens: 86
Time taken: 4.179 seconds
Tokens per second: 20.58
Peak GPU memory: 0.445 GB

Generated response:
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain artificial intelligence in simple words.
assistant
Artificial Intelligence (AI) is a type of technology that allows machines to learn and perform tasks that typically require human intelligence, such as recognizing images or playing chess. AI can be used for a wide range of applications, from self-driving cars to language translation to personalized medicine. It's a rapidly evolving field with many potential uses and applications, but it has the potential to revolutionize industries and improve our lives in countless ways.


In [10]:
prompts = [
    "Explain artificial intelligence in simple words.",
    "What is machine learning? Give three real-world examples.",
    "Explain the difference between supervised and unsupervised learning.",
    "Write a Python function to check whether a number is prime.",
    "What is the difference between a Python list and a tuple?",
    "A shop gives a 20% discount on an item priced at 1500 rupees. What is the final price?",
    "Explain overfitting in machine learning and how to prevent it.",
    "What is the purpose of an API in software development?",
    "Summarize the importance of data preprocessing in machine learning.",
    "Explain how a neural network learns from training data."
]

answers_4bit = []

for i, prompt in enumerate(prompts, 1):
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to("cuda:0")

    with torch.inference_mode():
        output = model_4bit.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    answer = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    answers_4bit.append(answer)

    print(f"\nPROMPT {i}: {prompt}")
    print("4-BIT ANSWER:", answer)

print("\nAll 10 4-bit answers generated successfully.")


PROMPT 1: Explain artificial intelligence in simple words.
4-BIT ANSWER: Artificial Intelligence (AI) is a type of technology that allows machines to learn and perform tasks that typically require human intelligence, such as recognizing images or playing chess. AI can be used for a wide range of applications, from self-driving cars to language translation to personalized medicine. It's a rapidly evolving field with many potential uses and applications, but it has the potential to revolutionize industries and improve our lives in countless ways.

PROMPT 2: What is machine learning? Give three real-world examples.
4-BIT ANSWER: Machine learning is a type of artificial intelligence (AI) that enables computers to learn from data and improve their performance over time without being explicitly programmed. It involves the use of algorithms and statistical models to enable machines to make decisions or predictions based on patterns in large amounts of data.

Here are three real-world example

In [11]:
import json

with open("/kaggle/working/task3_quality_4bit.json", "w", encoding="utf-8") as f:
    json.dump(
        [
            {"prompt": prompt, "answer": answer}
            for prompt, answer in zip(prompts, answers_4bit)
        ],
        f,
        indent=4,
        ensure_ascii=False
    )

print("4-bit quality results saved successfully.")

4-bit quality results saved successfully.


In [14]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Allocated memory:", round(torch.cuda.memory_allocated(0) / 1024**3, 3), "GB")
    print("Reserved memory:", round(torch.cuda.memory_reserved(0) / 1024**3, 3), "GB")

print("model_4bit exists:", "model_4bit" in globals())
print("tokenizer exists:", "tokenizer" in globals())
print("prompts exist:", "prompts" in globals())

CUDA available: True
GPU: Tesla T4
Allocated memory: 0.181 GB
Reserved memory: 0.951 GB
model_4bit exists: False
tokenizer exists: True
prompts exist: True


In [15]:
import torch
from transformers import AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(0)

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cuda:0"
)

model_fp16.eval()

print("FP16 model loaded successfully!")
print(
    "Allocated GPU memory:",
    round(torch.cuda.memory_allocated(0) / 1024**3, 3),
    "GB"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

FP16 model loaded successfully!
Allocated GPU memory: 0.929 GB


In [16]:
answers_fp16 = []

for i, prompt in enumerate(prompts, 1):
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to("cuda:0")

    with torch.inference_mode():
        output = model_fp16.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    answer = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    answers_fp16.append(answer)

    print(f"\nPROMPT {i}: {prompt}")
    print("FP16 ANSWER:", answer)

print("\nAll 10 FP16 answers generated successfully.")


PROMPT 1: Explain artificial intelligence in simple words.
FP16 ANSWER: Artificial Intelligence (AI) is a field of computer science that focuses on creating intelligent machines that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and understanding language. AI systems use algorithms and statistical models to analyze data and make predictions or decisions based on patterns and relationships within that data.

In simpler terms, AI involves the development of software programs that can learn from experience, recognize patterns, and respond to inputs in ways that mimic human behavior. This technology has

PROMPT 2: What is machine learning? Give three real-world examples.
FP16 ANSWER: Machine learning is a type of artificial intelligence that enables computers to learn and improve from data without being explicitly programmed. It involves the development of algorithms that can analyze patterns in large amounts of data and m

In [17]:
import json

with open("/kaggle/working/task3_quality_fp16.json", "w", encoding="utf-8") as f:
    json.dump(
        [
            {"prompt": prompt, "answer": answer}
            for prompt, answer in zip(prompts, answers_fp16)
        ],
        f,
        indent=4,
        ensure_ascii=False
    )

print("FP16 quality results saved successfully.")

FP16 quality results saved successfully.


In [18]:
print(
    "FP16 peak GPU memory:",
    round(torch.cuda.max_memory_allocated(0) / 1024**3, 3),
    "GB"
)

FP16 peak GPU memory: 0.934 GB
